# Example 03: Regional top non-Hermitian retarded bath (minimal skin-effect notebook)

This notebook keeps the **regional / experimental RHS** architecture and focuses on the effective non-Hermitian operator
\(H_\mathrm{eff} = H_\mathrm{device} + \Sigma^R_\mathrm{top}\) as the primary skin-effect diagnostic.


## Imports


In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__))))

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using Statistics
using PyPlot


## User-editable parameters
Small `Nx, Ny` are useful for quick checks. Quasi-1D choices like `Ny = 1 or 2` with larger `Nx` are useful for wire-like non-Hermitian comparisons.


In [ ]:
# Preset: fast checks vs larger research-sized test.
quick_test = true
run_time_evolution = true

# Geometry (keep these directly editable).
if quick_test
    Nx, Ny = 5, 3
    tmax, dt = 20.0, 1.0
else
    Nx, Ny = 10, 2
    tmax, dt = 40.0, 0.5
end

# Model parameters.
Nσ = 2
N_orb = 1
γ = 1.0
γso = 0.50 + 0.0im
μ = 0.0
Bz = 0.15

# Lead / pole parameters.
β = 33.0
N_λ1 = 49
N_λ2 = 20

# Top retarded bath parameters.
gamma0 = 0.20
gammay_caseA = 0.0
gammay_caseB = 0.10

# ODE solver controls (only used when run_time_evolution=true).
reltol = 1e-6
abstol = 1e-8

# Local plotting palette (restrained academic style).
PALETTE = Dict(
    :navy => "#1f3b73",
    :teal => "#2a9d8f",
    :cyan => "#5bc0be",
    :gold => "#e9c46a",
    :orange => "#f4a261",
    :redorange => "#e76f51",
    :lightbg => "#f7f8fa",
)

println("Nx=$(Nx), Ny=$(Ny), quick_test=$(quick_test), run_time_evolution=$(run_time_evolution)")
println("Cases: gammay_A=$(gammay_caseA), gammay_B=$(gammay_caseB)")
println("Sanity check gamma0 >= |gammay_caseB| : ", gamma0 >= abs(gammay_caseB))


## Geometry and indexing


In [ ]:
@inline local_index(x::Int, y::Int, Ny::Int) = (x - 1) * Ny + y

@inline function local_subrange(site::Int, N_loc::Int)
    a = (site - 1) * N_loc + 1
    b = site * N_loc
    return a:b
end

function top_edge_coupled_indices(; Nx::Int, Ny::Int, N_loc::Int)
    top_sites = [local_index(x, Ny, Ny) for x in 1:Nx]
    idx_couple = Int[]
    for s in top_sites
        append!(idx_couple, collect(local_subrange(s, N_loc)))
    end
    return top_sites, idx_couple
end

function site_scalar_to_grid(values::AbstractVector, Nx::Int, Ny::Int)
    @assert length(values) == Nx * Ny
    [values[local_index(x, y, Ny)] for y in Ny:-1:1, x in 1:Nx]
end


## Device Hamiltonian


In [ ]:
function add_uniform_onsite!(H::AbstractMatrix{ComplexF64}; Nx::Int, Ny::Int, N_loc::Int, μ::Float64, Bz::Float64)
    σz = ComplexF64[1 0; 0 -1]
    onsite = (-μ) * Matrix{ComplexF64}(I, N_loc, N_loc) + Bz * σz
    for site in 1:(Nx * Ny)
        r = local_subrange(site, N_loc)
        H[r, r] .+= onsite
    end
    return H
end


## Standard left/right leads


In [ ]:
function build_two_terminal_blocks(p_model::ModelParamsTDNEGF; β::Float64, N_λ1::Int, N_λ2::Int)
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)

    Σᴸ_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    Σᴳ_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    χ_nλ = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                      p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)

    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=p_model.Nx, y_coup=1:p_model.Ny)
    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=1,          y_coup=1:p_model.Ny)

    left_block = SelfEnergyBlock(:left, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anR)
    return [left_block, right_block]
end


## Regional top non-Hermitian retarded bath


In [ ]:
function build_top_bath_ΣR(; n_coupled_sites::Int, N_loc::Int, gamma0::Float64, gammay::Float64)
    @assert N_loc == 2 "This notebook assumes one orbital and spin-1/2 (N_loc=2)."
    @assert gamma0 >= 0.0 "gamma0 should be non-negative."
    if gamma0 < abs(gammay)
        @warn "gamma0 < |gammay|: one spin channel can become amplifying."
    end

    σy = ComplexF64[0 -im; im 0]
    Σ_site = -1im .* (gamma0 .* Matrix{ComplexF64}(I, N_loc, N_loc) .+ gammay .* σy)
    ΣR_loc = kron(Matrix{ComplexF64}(I, n_coupled_sites, n_coupled_sites), Σ_site)
    return Σ_site, ΣR_loc
end

function embed_local_block(idx_couple::Vector{Int}, block_local::AbstractMatrix{ComplexF64}, Nglobal::Int)
    Σ_global = zeros(ComplexF64, Nglobal, Nglobal)
    Σ_global[idx_couple, idx_couple] .= block_local
    return Σ_global
end

function make_case(; Nx::Int, Ny::Int, Nσ::Int, N_orb::Int,
                   γ::Float64, γso, β::Float64, N_λ1::Int, N_λ2::Int,
                   μ::Float64, Bz::Float64,
                   gamma0::Float64, gammay::Float64)

    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb, Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    H_device = build_H_ab(; Nx, Ny, Nσ, N_orb, γ=γ, γso=γso)
    add_uniform_onsite!(H_device; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc, μ=μ, Bz=Bz)
    p_model.H_ab .= H_device
    p_model.H0_ab .= H_device

    blocks = build_two_terminal_blocks(p_model; β=β, N_λ1=N_λ1, N_λ2=N_λ2)
    Δ_blocks = ComplexF64[0.5 + 0.0im, -0.5 + 0.0im]

    top_sites, idx_couple = top_edge_coupled_indices(; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc)
    Σ_site, ΣR_loc = build_top_bath_ΣR(; n_coupled_sites=length(top_sites), N_loc=p_model.N_loc, gamma0=gamma0, gammay=gammay)
    Nglobal = size(H_device, 1)
    ΣR_top_global = embed_local_block(idx_couple, ΣR_loc, Nglobal)

    top_bath = TDNEGF.ExtraBathSpec(:nonhermitian_retarded, :top_edge_nonhermitian, idx_couple, ΣR_loc, true)
    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model, [top_bath])
    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)

    return (; p_model, p_blocks, u0, top_sites, idx_couple, Σ_site, ΣR_loc, ΣR_top_global, gammay)
end

caseA = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gammay_caseA)
caseB = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gammay_caseB)

println("Built Case A and Case B.")


In [ ]:
println("--- Concise sanity/debug checks ---")
println("gamma0 >= |gammay_caseA|: ", gamma0 >= abs(gammay_caseA))
println("gamma0 >= |gammay_caseB|: ", gamma0 >= abs(gammay_caseB))
println("top-edge site indices (A): ", caseA.top_sites)
println("idx_couple length / sample: ", length(caseA.idx_couple), " / ", caseA.idx_couple[1:min(end, 8)])
println("local ΣR shape: ", size(caseA.ΣR_loc))
println("embedded/global ΣR shape: ", size(caseA.ΣR_top_global))
println("affected sites (A): ", caseA.top_sites)
println("embedded/global ΣR shape (B): ", size(caseB.ΣR_top_global))


## Effective non-Hermitian Hamiltonian diagnostics
Core objective: compare `gammay = 0` vs `gammay ≠ 0` using the finite-device operator
\(H_\mathrm{eff}=H_\mathrm{device}+\Sigma^R_\mathrm{top}\).


In [ ]:
function heff(case_setup)
    Matrix(case_setup.p_model.H_ab) + case_setup.ΣR_top_global
end

function site_weights_from_state(vec_state::AbstractVector, Nx::Int, Ny::Int, N_loc::Int)
    weights = zeros(Float64, Nx * Ny)
    for site in 1:(Nx * Ny)
        r = local_subrange(site, N_loc)
        weights[site] = real(sum(abs2, vec_state[r]))
    end
    s = sum(weights)
    return s > 0 ? weights ./ s : weights
end

function select_mode_indices(vals; k_each=2)
    ord = sortperm(imag.(vals))
    low = ord[1:min(k_each, end)]
    high = ord[max(1, end-k_each+1):end]
    return unique(vcat(low, high))
end

function plot_heff_diagnostics(case_setup; mode_k_each=2)
    H = heff(case_setup)
    ev = eigen(H)
    vals, vecs = ev.values, ev.vectors

    fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=true)
    axs[1].set_facecolor(PALETTE[:lightbg])
    axs[1].scatter(real.(vals), imag.(vals), s=26, c=PALETTE[:navy], alpha=0.85, edgecolors="none")
    axs[1].axhline(0.0, color=PALETTE[:orange], lw=1.0, alpha=0.8)
    axs[1].set_xlabel("Re(E)")
    axs[1].set_ylabel("Im(E)")
    axs[1].set_title("Complex spectrum, gammay=$(case_setup.gammay)")
    axs[1].grid(alpha=0.25)

    mids = select_mode_indices(vals; k_each=mode_k_each)
    axs[2].set_facecolor(PALETTE[:lightbg])
    for (j, idx) in enumerate(mids)
        w = site_weights_from_state(vecs[:, idx], case_setup.p_model.Nx, case_setup.p_model.Ny, case_setup.p_model.N_loc)
        axs[2].plot(1:length(w), w, lw=1.8,
                    color=(j % 2 == 1 ? PALETTE[:teal] : PALETTE[:gold]),
                    label="mode $(idx), Im(E)=$(round(imag(vals[idx]), digits=3))")
    end
    axs[2].set_xlabel("site index")
    axs[2].set_ylabel("normalized right-mode weight")
    axs[2].set_title("Selected right-eigenvector weights")
    axs[2].grid(alpha=0.25)
    axs[2].legend(frameon=false, fontsize=8)
    plt.show()

    # Lattice-resolved maps for one strongly damped and one least-damped mode.
    map_ids = [argmin(imag.(vals)), argmax(imag.(vals))]
    fig2, ax2 = plt.subplots(1, 2, figsize=(10.8, 3.9), constrained_layout=true)
    for (j, idx) in enumerate(map_ids)
        w = site_weights_from_state(vecs[:, idx], case_setup.p_model.Nx, case_setup.p_model.Ny, case_setup.p_model.N_loc)
        im = ax2[j].imshow(site_scalar_to_grid(w, case_setup.p_model.Nx, case_setup.p_model.Ny),
                           cmap="cividis", aspect="auto")
        ax2[j].set_title("mode $(idx), Im(E)=$(round(imag(vals[idx]), digits=3))")
        ax2[j].set_xlabel("x")
        ax2[j].set_ylabel("y (top at top)")
        plt.colorbar(im, ax=ax2[j], fraction=0.046)
    end
    fig2.suptitle("Lattice-resolved right-mode weights, gammay=$(case_setup.gammay)")
    plt.show()

    return vals
end

valsA = plot_heff_diagnostics(caseA)
valsB = plot_heff_diagnostics(caseB)

println("Mean Im(E): Case A = $(mean(imag.(valsA))), Case B = $(mean(imag.(valsB)))")


## Optional time-evolution observables
Secondary diagnostics only: one final-time spin-density map and compact lead spin-current comparison.


In [ ]:
function run_case(case_setup; tspan=(0.0, 20.0), reltol=1e-6, abstol=1e-8, dt_save=1.0)
    p_model, p_blocks, u0 = case_setup.p_model, case_setup.p_blocks, case_setup.u0

    save_times = collect(range(tspan[1], tspan[2]; step=dt_save))
    prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)
    sol = solve(prob, Vern7(); adaptive=true, saveat=save_times, dense=false, reltol=reltol, abstol=abstol)

    obs = ObservablesTDNEGF(p_model; N_tmax=length(sol.t), N_leads=length(p_blocks.blocks))
    obs.t = sol.t

    for (it, ut) in enumerate(sol.u)
        obs.idx = it
        ptr = pointer_blocks(ut, p_blocks.dims_ρ_ab, p_blocks.aux_layout)
        obs_n_i!(ptr, p_model, obs)
        obs_σ_i!(ptr, p_model, obs)
        obs_Ixα!(ptr, p_blocks, obs)
    end

    return sol, obs
end

function final_spin_map(obs::ObservablesTDNEGF, comp::Int, Nx::Int, Ny::Int)
    vals = vec(obs.σx_i[:, comp, end])
    return site_scalar_to_grid(vals, Nx, Ny), vals
end

if run_time_evolution
    tspan = (0.0, tmax)
    solA, obsA = run_case(caseA; tspan=tspan, reltol=reltol, abstol=abstol, dt_save=dt)
    solB, obsB = run_case(caseB; tspan=tspan, reltol=reltol, abstol=abstol, dt_save=dt)

    # Final-time spin-density map (s_z) and compact left-right current summary.
    mapA, szA = final_spin_map(obsA, 3, Nx, Ny)
    mapB, szB = final_spin_map(obsB, 3, Nx, Ny)

    fig, axs = plt.subplots(1, 3, figsize=(14.2, 4.0), constrained_layout=true)
    vmax = max(maximum(abs.(mapA)), maximum(abs.(mapB))) + 1e-12

    imA = axs[1].imshow(mapA, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    axs[1].set_title("Case A: final s_z")
    axs[1].set_xlabel("x"); axs[1].set_ylabel("y")

    imB = axs[2].imshow(mapB, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    axs[2].set_title("Case B: final s_z")
    axs[2].set_xlabel("x"); axs[2].set_ylabel("y")

    # Lead-resolved final spin current components.
    labels = ["Sx", "Sy", "Sz"]
    x = 1:3
    IA = [obsA.Iαx[1, c, end] - obsA.Iαx[2, c, end] for c in 1:3]
    IB = [obsB.Iαx[1, c, end] - obsB.Iαx[2, c, end] for c in 1:3]
    w = 0.35
    axs[3].bar(x .- w/2, real.(IA), width=w, color=PALETTE[:teal], label="A: gammay=0")
    axs[3].bar(x .+ w/2, real.(IB), width=w, color=PALETTE[:orange], label="B: gammay=$(caseB.gammay)")
    axs[3].set_xticks(collect(x)); axs[3].set_xticklabels(labels)
    axs[3].set_ylabel("Re(I_L - I_R) at final time")
    axs[3].set_title("Lead spin-current asymmetry")
    axs[3].grid(alpha=0.25)
    axs[3].legend(frameon=false, fontsize=8)

    plt.colorbar(imB, ax=[axs[1], axs[2]], fraction=0.046, label="s_z")
    plt.show()

    top_idx = [local_index(ix, Ny, Ny) for ix in 1:Nx]
    bot_idx = [local_index(ix, 1, Ny) for ix in 1:Nx]
    Δtopbot_A = mean(szA[top_idx]) - mean(szA[bot_idx])
    Δtopbot_B = mean(szB[top_idx]) - mean(szB[bot_idx])
    println("Final Δs_z(top-bottom): Case A = $(Δtopbot_A), Case B = $(Δtopbot_B)")
else
    println("Time-evolution section skipped (run_time_evolution=false).")
end


## Conclusions

- Turning on `gammay` changes the non-Hermitian retarded bath from spin-independent damping to spin-selective damping at the top edge.
- The primary diagnostics here are the complex spectrum and right-eigenvector weight maps of `H_eff`, which directly reveal changes in mode damping and boundary accumulation trends.
- Final-time spin density and lead spin-current asymmetry are kept as compact supporting observables.
- Out of scope: full non-Hermitian bath consistency with lesser/greater components (this notebook intentionally remains **retarded-only**).
